In [1]:
import pyspark
import time
from pyspark import SparkContext

In [2]:
sc = SparkContext(master='local[2]')

In [3]:
sc

<SparkContext master=local[2] appName=pyspark-shell>

In [4]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Random Forest Classifier").getOrCreate()

In [5]:
start_time1 = time.time()
true_df = spark.read.csv("True.csv", inferSchema=True, header=True)
fake_df = spark.read.csv("Fake.csv", inferSchema=True, header=True)
end_time1 = time.time()

In [6]:
true_df.show(5)

+--------------------+--------------------+------------+------------------+
|               title|                text|     subject|              date|
+--------------------+--------------------+------------+------------------+
|As U.S. budget fi...|WASHINGTON (Reute...|politicsNews|December 31, 2017 |
|U.S. military to ...|WASHINGTON (Reute...|politicsNews|December 29, 2017 |
|Senior U.S. Repub...|WASHINGTON (Reute...|politicsNews|December 31, 2017 |
|FBI Russia probe ...|WASHINGTON (Reute...|politicsNews|December 30, 2017 |
|Trump wants Posta...|SEATTLE/WASHINGTO...|politicsNews|December 29, 2017 |
+--------------------+--------------------+------------+------------------+
only showing top 5 rows



In [7]:
fake_df.show(5)

+--------------------+--------------------+-------+-----------------+
|               title|                text|subject|             date|
+--------------------+--------------------+-------+-----------------+
| Donald Trump Sen...|Donald Trump just...|   News|December 31, 2017|
| Drunk Bragging T...|House Intelligenc...|   News|December 31, 2017|
| Sheriff David Cl...|On Friday, it was...|   News|December 30, 2017|
| Trump Is So Obse...|On Christmas day,...|   News|December 29, 2017|
| Pope Francis Jus...|Pope Francis used...|   News|December 25, 2017|
+--------------------+--------------------+-------+-----------------+
only showing top 5 rows



In [8]:
from pyspark.sql.functions import lit, rand

In [9]:
start_time2 = time.time()
data= true_df.withColumn('trueOrfake', lit(1)).union(fake_df.withColumn('trueOrfake', lit(0))).orderBy(rand())
end_time2 = time.time()

In [10]:
data = data.select(["title", "subject", 'trueOrfake'])

In [11]:
desired_values = ["politicsNews", "News", "politics", "Government News", "left-news", "US_News", "Middle-east", "worldnews"]

In [12]:
from pyspark.sql.functions import col

In [13]:
start_time3 = time.time()
df_filtered = data.filter(col("subject").isin(desired_values))
end_time3 = time.time()

In [14]:
df_filtered.groupby('subject').count().sort("count",ascending=False).show()

+---------------+-----+
|        subject|count|
+---------------+-----+
|   politicsNews|11209|
|      worldnews|10115|
|           News| 8501|
|       politics| 6525|
|      left-news| 4216|
|Government News| 1543|
|        US_News|  767|
|    Middle-east|  762|
+---------------+-----+



In [15]:
df_filtered.toPandas()['title'].isnull().sum()

0

In [16]:
df_filtered.toPandas()['subject'].isnull().sum()

0

In [17]:
df_filtered.show(5)

+--------------------+------------+----------+
|               title|     subject|trueOrfake|
+--------------------+------------+----------+
| WTF: Top Trump A...|        News|         0|
|State prosecutors...|politicsNews|         1|
|Obamacare whiplas...|politicsNews|         1|
|Sickening Reason ...|    politics|         0|
|South Sudan oppos...|   worldnews|         1|
+--------------------+------------+----------+
only showing top 5 rows



In [18]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover, CountVectorizer, IDF, StringIndexer

In [19]:
# Stages for the pipeline
start_time4 = time.time()
tokenizer = Tokenizer(inputCol='title', outputCol='mytokens')
stopwordRemover = StopWordsRemover(inputCol='mytokens',outputCol='filtered_tokens')
vectorizer = CountVectorizer(inputCol='filtered_tokens',outputCol='rawFeatures')
idf = IDF(inputCol='rawFeatures', outputCol='vectorizedFeatures')
labelEncoder = StringIndexer(inputCol='subject',outputCol='label').fit(df_filtered)
end_time4 = time.time()

In [20]:
labelEncoder.transform(df_filtered).show(5)

+--------------------+------------+----------+-----+
|               title|     subject|trueOrfake|label|
+--------------------+------------+----------+-----+
| WTF: Top Trump A...|        News|         0|  2.0|
|State prosecutors...|politicsNews|         1|  0.0|
|Obamacare whiplas...|politicsNews|         1|  0.0|
|Sickening Reason ...|    politics|         0|  3.0|
|South Sudan oppos...|   worldnews|         1|  1.0|
+--------------------+------------+----------+-----+
only showing top 5 rows



In [21]:
start_time5 = time.time()
df_filtered = labelEncoder.transform(df_filtered)
end_time5 = time.time()

In [22]:
(train_df, test_df) = df_filtered.randomSplit((0.7,0.3),seed=42)

In [23]:
train_df.count()

30733

In [24]:
test_df.count()

12905

In [25]:
train_df.show(2)

+--------------------+-------+----------+-----+
|               title|subject|trueOrfake|label|
+--------------------+-------+----------+-----+
| #AfterTrumpImplo...|   News|         0|  2.0|
| #BlackLivesMatte...|   News|         0|  2.0|
+--------------------+-------+----------+-----+
only showing top 2 rows



In [26]:
# machine learning model (Estimator) (data to model)
from pyspark.ml.classification import RandomForestClassifier
rf = RandomForestClassifier(labelCol = 'trueOrfake',featuresCol='vectorizedFeatures', numTrees=1000)

In [27]:
from pyspark.ml import Pipeline

In [28]:
pipeline = Pipeline(
    stages=[tokenizer, stopwordRemover, vectorizer, idf, rf]
)

In [29]:
pipeline.stages

Param(parent='Pipeline_cca9b75c2d63', name='stages', doc='a list of pipeline stages')

In [30]:
start_time6 = time.time()
rf_model = pipeline.fit(train_df)
end_time6 = time.time()

In [31]:
rf_model

PipelineModel_d5fa13249c06

In [32]:
#testare
start_time7 = time.time()
predictions = rf_model.transform(test_df)
end_time7 = time.time()

In [33]:
predictions.columns

['title',
 'subject',
 'trueOrfake',
 'label',
 'mytokens',
 'filtered_tokens',
 'rawFeatures',
 'vectorizedFeatures',
 'rawPrediction',
 'probability',
 'prediction']

In [34]:
predictions.select('vectorizedFeatures','rawPrediction', 'probability','subject','label','trueOrFake','prediction').show(5000)

+--------------------+--------------------+--------------------+---------------+-----+----------+----------+
|  vectorizedFeatures|       rawPrediction|         probability|        subject|label|trueOrFake|prediction|
+--------------------+--------------------+--------------------+---------------+-----+----------+----------+
|(35644,[1,79,114,...|[532.743609141725...|[0.53274360914172...|           News|  2.0|         0|       0.0|
|(35644,[0,1,145,2...|[525.016780526055...|[0.52501678052605...|           News|  2.0|         0|       0.0|
|(35644,[1,306,717...|[530.079262011906...|[0.53007926201190...|           News|  2.0|         0|       0.0|
|(35644,[1,544,109...|[526.523492889899...|[0.5265234928899,...|           News|  2.0|         0|       0.0|
|(35644,[0,1,74,28...|[523.807277263300...|[0.52380727726330...|           News|  2.0|         0|       0.0|
|(35644,[1,85,88,4...|[528.844521295170...|[0.52884452129517...|           News|  2.0|         0|       0.0|
|(35644,[1,13,126,.

In [35]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

In [36]:
evaluator = MulticlassClassificationEvaluator(predictionCol='prediction',labelCol='trueOrfake')

In [41]:
accuracy = evaluator.evaluate(predictions)
accuracy*100

77.82106935778445


In [39]:
bin_eval = BinaryClassificationEvaluator(rawPredictionCol='prediction', labelCol='trueOrfake')

In [42]:
roc = bin_eval.evaluate(predictions)
print(roc*100)

78.48453678162018


In [49]:
auPR = bin_eval.evaluate(predictions, {bin_eval.metricName: "areaUnderPR"})
print(auPR*100)

87.92562943638957


In [51]:
evaluator.setMetricName('truePositiveRateByLabel')
evaluator.setMetricLabel(1.0)
print(evaluator.evaluate(predictions))

0.5799526440410419


In [52]:
evaluator.setMetricName('truePositiveRateByLabel')
evaluator.setMetricLabel(0.0)
print(evaluator.evaluate(predictions))

0.9897380915913616


In [53]:
evaluator.setMetricName('falsePositiveRateByLabel')
evaluator.setMetricLabel(1.0)
print(evaluator.evaluate(predictions))

0.010261908408638382


In [54]:
evaluator.setMetricName('falsePositiveRateByLabel')
evaluator.setMetricLabel(0.0)
print(0.42004735595895815)

0.42004735595895815


In [48]:
execution_time1 = end_time1 - start_time1 #timp de executie pentru citirea datelor
execution_time2 = end_time2 - start_time2 #timp de executie pentru combinarea datelor si amestecarea lor (preprocesarea datelor)
execution_time3 = end_time3 - start_time3 #timp de executie pentru pastrarea valorilor corecte pentru coloana subject (preprocesarea datelor)
execution_time4 = end_time4 - start_time4 #timp de executie pentru procesarea datelor
execution_time5 = end_time5 - start_time5 #timp de executie pentru procesarea datelor 

execution_time6 = end_time6 - start_time6 #timp de executie pentru antrenarea modelului
execution_time7 = end_time7 - start_time7 #timp de executie pentru testare

total_execution_time_for_preprocessing_data = execution_time2 + execution_time3
total_execution_time_for_processing_data = execution_time4 + execution_time5
total_execution_time_for_testing = execution_time7

print("Timp de executie pentru citirea datelor: {} secunde".format(execution_time1))
print("Timp de executie pentru preprocesarea datelor: {} secunde".format(total_execution_time_for_preprocessing_data))
print("Timp de executie pentru procesarea datelor: {} secunde".format(total_execution_time_for_processing_data))
print("Timp de executie pentru antrenarea modelului: {} secunde".format(execution_time6))
print("Timp de executie pentru testare: {} secunde".format(total_execution_time_for_testing))

Timp de executie pentru citirea datelor: 7.753133535385132 secunde
Timp de executie pentru preprocesarea datelor: 0.10422658920288086 secunde
Timp de executie pentru procesarea datelor: 2.0737528800964355 secunde
Timp de executie pentru antrenarea modelului: 242.47225165367126 secunde
Timp de executie pentru testare: 0.19728899002075195 secunde
